# Spiegazione script Pix2Pix+
Un interessante articolo sulle [CNN](https://developersbreach.com/convolution-neural-network-deep-learning/).  

Dopo i risultati ottenuti con `Pix2Pix.py` abbiamo provato a migliorare la rete optando sempre per una rete di tipo **GAN (Generative Adversarial Network)**, quindi composta da due sottoreti neurali: **UNet 3+** per il **Generatore** e **PatchGAN** per il **Discriminatore**. L'architettura si compone, per l'appunto, di un Generatore che trasformi immagini **label\_free** in versioni **stained** e di un Discriminatore che valuta se le immagini prodotte dal generatore siano realistiche confrontandole con le immagini target.  
Le due reti si allenano in competizione: il generatore cerca di produrre immagini sempre più simili al ground truth (il dato reale), mentre il discriminatore cerca di riconoscere quelle false, migliorando così la qualità finale.

Per definire le reti Generatore e Discriminatore dobbiamo dichiarare delle classi, con tutti i suoi costrutti. Ora inizieremo ad analizzare il Generatore

In [ ]:
import torch.nn as nn

class UNet3PlusGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, base_filter=64):
        super(UNet3PlusGenerator, self).__init__()

Quello definito sopra è la dichiarazione base della classe `UNet3PlusGenerator`, la quale eredita i metodi della classe padre `nn.Module`. Di seguito, all'interno del costruttore della classe troviamo una serie di parametri:  
- `in_channels=3`: descrive il numero di canali dell'immagine in ingresso, sono 3 per rappresentare i canali RGB;  
- `out_channels=3`: descrive il numero di canali dell'immagine in uscita, anch'essi sono 3 perchè vogliamo come output un'immagine RGB;  
- `base_filter=64`: descrive il numero di filtri nei layer convoluzionali;  

**Nota bene**: in che senso _"filtri nei layer convoluzionali"?_ Un layer convoluzionale è uno strato delle rete neurale che applica una convoluzione (operazione matematica che combina una piccola finestra, filtro, con porzioni locali di un'immagine correndo pixel per pixel) per estrarre caratteristiche locali e riconoscere bordi; un filtro (o kernel) invece è una matrice di pesi che scorre (convoluzione) sull'immagine per estrarre caratteristiche. Se un layer ha 64 filtri, significa che ogni filtro analizza la stessa immagine in modo diverso e che l'output del layer sarà una **feature map** con 64 canali.  

Nelle Reti Neurali **Convoluzionali (CNN)**, i neuroni sono l’uscita (attivazione) risultante dall'applicazione del filtro su una porzione dell’immagine. Quindi ogni neurone corrisponde a un valore della feature map. Se prendessimo ad esempio un'immagine 64x64 in grayscale (quindi 1 solo canale di colore) e ci applicassimo sopra 32 filtri 3x3 otterremo una feature map 64x64x32, in qui avrei 4096 (64\*64) neuroni per filtro, per un totale di 131072 (4096\*32) neuroni per quel layer.  
Nell'esempio di poco fa era presente un filtro 3x3, ma sarebbe cambiato qualcosa se fosse cambiata la dimensione del filtro? La risposta è si, ma in funzione di due parametri molto importanti, ovvero il **padding** e lo **stride**.
- Il primo indica il numero di bordi artificiali attorno all'immagine con valori impostati a 0 per controllare la dimensione della feature map in uscita;  
- Il secondo indica di quanti pixel il filtro si muove;   

La **dimensione spaziale** \(larghezza o altezza\) della feature map in uscita da un layer si calcola come $Output=\lfloor\frac{(N+2P-K)}{S}+1\rfloor$ dove:
- N rappresenta la dimensione in input (noi lavoreremo con quadrati quindi non faremo differenze)
- P rappresenta il valore del padding
- K rappresenta il valore del kernel size
- S rappresenta il valore dello stride

La forma della feature map finale sarà un **tensore** (una struttura dati che rappresenta array multidimensionali; può avere 0, 1, 2 o più dimensioni es: uno scalare è un tensore 0D, un vettore è 1D, una matrice è 2D, e così via) di forma $(N_{Filter},\ \lfloor\frac{(N+2P-K)}{S}+1\rfloor,\ \lfloor\frac{(N+2P-K)}{S}+1\rfloor)$.  

Ora che abbiamo spiegato il funzionamento del padding si potrebbe pensare che impostare un grande valore di padding possa essere una buona idea perché ci permetterebbe di avere una dimensione spaziale della feature map maggiore, e di conseguenza più neuroni, ma in realtà non è così. Questa conclusione è dovuta al fatto che un padding eccessivamente alto possa gonfiare con dati fittizzi la feature map, aumentandone la dimensione, e i tempi dei calcoli su essa basata, senza però star aumentando realmente la quantità di informazione.

In [ ]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

Sopra è riportata la definizione della funzione `conv_block` la quale appartiene ai metodi della classe `UNet3PlusGenerator`. Tale funzione, quando richiamata esegue, in ordine sequenziale:  
1. `Conv2d`: esegue la convoluzione seguendo i parametri forniti (con `padding=1` otteniamo una dimesnione spaziale della feature map pari a 512, ovvero la dimensione spaziale delle nostre immagini di input);  
2. `BatchNorm2d`: applica una normalizzazione sulle attivazioni (numero di neuroni) canale per canale. Stabilizza l'apprendimento e fa convergere il modello più velocemente;  
3. `ReLU`: impone non linearità e con il parametro `implace=True` sovrascrive il tensore in ingresso;  

**Nota bene**: _"Perchè la normalizzazione aiuta a stabilizzare l'apprendimento e fa convergere il modello più velocemente"?_ BatchNorm calcola la media e la deviazione standard per ogni canale, considerando tutti i pixel di quel canale nell'intero batch. I valori vengono normalizzati e poi ri-scalati tramite due parametri appresi, $\gamma$ e $\beta$, che permettono alla rete di mantenere flessibilità e capacità espressiva. La formula è: $BN(x)=\gamma\cdot\frac{x-\mu}{\sigma}+\beta$.  
Definiamo ora il concetto di **batch**, ovvero il gruppo di esempi (nel nostro caso immagini) elaborati insieme in una singola iterazione di training e usati per aggiornare i pesi.

In [ ]:
batch_size = 8

Supponiamo ora di avere un batch di immagini di shape: $(B,\ C,\ H,\ W)$ dove:  
- B è il numero di immagini del batch;  
- C è il numero di canali;  
- H, W sono altezza e larghezza delle immagini;  

Per ogni canale $c$ `BatchNorma2d` calcola:
- la media $\mu_c$ di tutti i valori in quel canale, per tutte le immagini del batch: $\mu_c=mean(x[b,\ c,\ h,\ w])\hspace{1em}\forall b,\ h, w$;  
- la deviazione standard $\sigma_c$ degli stessi valori: $\sigma_c=std((x[b,\ c,\ h,\ w]))$
- γ e β sono parametri appresi dalla rete, proprio come i pesi dei layer. Inizialmente: $\gamma=1$ (moltiplica il risultato normalizzato, quindi lo lascia com’è) e $\beta=0$ (non lo sposta).  

Quindi: `BatchNorma2d` prende il batch di feature map e normalizza i valori canale per canale, per ogni batch, prima di passarli al layer successivo.

Durante il training, ogni layer creerà nuove rappresentazioni dei canali e ogni batch passerà per gli **encoder** che eseguiranno tali operazioni. Al primo encorder i canali in ingresso, e quindi quelli su cui verrà calcolato `BatchNorm2d` sono 3 (RGB), dopo aumenteranno in funzione di ciò che abbiamo inserito nel codice

In [ ]:
self.enc1 = conv_block(in_channels=3, out_channels=64)
self.enc2 = conv_block(in_channels=64, out_channels=128)

`ReLU` è una funzione di attivazione non lineare che annulla i valori negativi e lascia invariati quelli positivi: $ReLU(x)=\begin{cases}x \hspace{1em} se \ x > 0 \\ 0 \hspace{1em} se \ x \leq 0\end{cases}$.  
La `ReLU` interrompe la continuità lineare applicando una regola non lineare: annulla alcune attivazioni e ne mantiene altre invariate. Introduce non linearità nella rete, permettendo di apprendere relazioni complesse nei dati.  
Senza ReLU (o attivazioni simili), la rete rimarrebbe equivalente a una singola trasformazione lineare, anche con molti layer.

In [ ]:
self.enc1 = conv_block(in_channels=3, out_channels=64)  # self.shape=(batch_size, 3, 512, 512) -> self.shape=(batch_size, 64, 512, 512)
self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # self.shape=(batch_size, 64, 256, 256)